# Minimal word-segmentation probe: M00–M30

Choose **Runtime → Run all**. When prompted, upload the Phase 1 output ZIP containing exactly 30 checkpoints. Everything runs in `/content`; Google Drive is not mounted or used.

In [ ]:
# 1. Clone the public repository and install the exact analysis code
import pathlib, shutil, subprocess, sys

REPOSITORY = 'https://github.com/ss-sebastian/developmental_checkpoints_word_recognition.git'
PROJECT = pathlib.Path('/content/developmental_checkpoints_word_recognition')
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(PROJECT)], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PROJECT),
    'datasets>=3.0', 'pandas>=2.0', 'matplotlib>=3.8'
], check=True)
# A package installed by a subprocess is not always added to the already-running
# Colab kernel's import path. Add the src-layout explicitly and verify it now.
SOURCE_DIR = PROJECT / 'src'
if str(SOURCE_DIR) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIR))
import importlib
importlib.invalidate_caches()
import devlm
print('devlm imported from:', pathlib.Path(devlm.__file__).resolve())
print('Installed repository commit:')
subprocess.run(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], check=True)

In [ ]:
# 2. Require CUDA so the run cannot silently fall back to CPU
import torch
assert torch.cuda.is_available(), 'CUDA is unavailable. Select Runtime > Change runtime type > T4 GPU, then Run all again.'
print('CUDA device:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

In [ ]:
# 3. Upload and validate the Phase 1 ZIP
from google.colab import files
import zipfile

print('Upload devlm_phase1_outputs.zip (30 checkpoints + session_split.json + ipa_feature_mapping.json)')
uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one Phase 1 ZIP, then rerun this cell.')
zip_name, zip_bytes = next(iter(uploaded.items()))
if not zip_name.lower().endswith('.zip'):
    raise ValueError('The uploaded file must be a .zip archive.')
zip_path = pathlib.Path('/content') / pathlib.Path(zip_name).name
zip_path.write_bytes(zip_bytes)
CHECKPOINT_ROOT = pathlib.Path('/content/devlm_phase1_checkpoints')
if CHECKPOINT_ROOT.exists():
    shutil.rmtree(CHECKPOINT_ROOT)
CHECKPOINT_ROOT.mkdir(parents=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(CHECKPOINT_ROOT)

from devlm.adaptation.train import discover_checkpoints, discover_feature_table
checkpoints = discover_checkpoints(CHECKPOINT_ROOT)
feature_table = discover_feature_table(CHECKPOINT_ROOT)
split_candidates = sorted(CHECKPOINT_ROOT.rglob('session_split.json'))
if len(split_candidates) != 1:
    raise ValueError(f'Expected one session_split.json; found {len(split_candidates)}')
SESSION_SPLIT = split_candidates[0]
print(f'Validated {len(checkpoints)} checkpoints: {checkpoints[0][0]}–{checkpoints[-1][0]}')
print('Session split:', SESSION_SPLIT)
print('Feature table:', feature_table)

In [ ]:
# 4. Download IPA-CHILDES and retain only Phase 1 validation sessions
# WORD_BOUNDARY is preserved here for labels; it will never enter the GRU.
import csv, json, math
from datasets import load_dataset

split_manifest = json.loads(SESSION_SPLIT.read_text())
validation_keys = {(str(a), str(b)) for a, b in split_manifest['validation']}
SOURCE_CSV = pathlib.Path('/content/ipa_childes_phase1_validation_with_boundaries.csv')
fieldnames = [
    'corpus_id', 'transcript_id', 'id', 'target_child_age',
    'ipa_transcription', 'processed_gloss', 'gloss'
]
rows = load_dataset(
    'phonemetransformers/IPA-CHILDES', 'EnglishNA',
    split='train', streaming=True
)
seen_sessions = set()
written = 0
with SOURCE_CSV.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    for scanned, row in enumerate(rows, 1):
        key = (str(row.get('corpus_id')), str(row.get('transcript_id')))
        age = row.get('target_child_age')
        if key in validation_keys and age is not None and math.isfinite(float(age)):
            writer.writerow({name: row.get(name, '') for name in fieldnames})
            seen_sessions.add(key)
            written += 1
        if scanned % 100_000 == 0:
            print(f'Scanned {scanned:,} rows; retained {written:,} rows from {len(seen_sessions):,}/{len(validation_keys):,} sessions', flush=True)
missing = validation_keys - seen_sessions
if missing:
    raise RuntimeError(f'IPA-CHILDES download is missing {len(missing)} Phase 1 validation sessions; first={sorted(missing)[0]}')
print(f'Ready: {written:,} utterances from all {len(seen_sessions):,} Phase 1 validation sessions')

In [ ]:
# 5. Run M00 and all 30 frozen developmental checkpoints
OUTPUT_ROOT = pathlib.Path('/content/word_segmentation_probe_outputs')
if OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)
command = [
    sys.executable, '-u', '-m', 'devlm.adaptation.segmentation_probe_cli',
    '--source-csv', str(SOURCE_CSV),
    '--checkpoint-root', str(CHECKPOINT_ROOT),
    '--session-split', str(SESSION_SPLIT),
    '--feature-table', str(feature_table),
    '--output-root', str(OUTPUT_ROOT),
    '--device', 'cuda'
]
print('Starting the complete 31-model probe...', flush=True)
subprocess.run(command, check=True)

In [ ]:
# 6. Display the requested developmental curve and onset result
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

metrics = pd.read_csv(OUTPUT_ROOT / 'summary' / 'checkpoint_metrics.tsv', sep='\t')
onset = json.loads((OUTPUT_ROOT / 'summary' / 'segmentation_onset.json').read_text())
display(metrics[[
    'checkpoint_id', 'checkpoint_hours', 'test_auc',
    'test_auc_ci_low', 'test_auc_ci_high',
    'test_balanced_accuracy', 'test_boundary_prevalence', 'test_accuracy'
]])
x = range(len(metrics))
plt.figure(figsize=(13, 5))
plt.plot(x, metrics.test_auc, marker='o', label='Test ROC-AUC')
plt.fill_between(x, metrics.test_auc_ci_low, metrics.test_auc_ci_high, alpha=.2, label='Session-bootstrap 95% CI')
plt.axhline(.5, color='black', linestyle='--', linewidth=1, label='Chance')
plt.xticks(list(x), metrics.checkpoint_id, rotation=90)
plt.ylim(0, 1)
plt.xlabel('Frozen model')
plt.ylabel('ROC-AUC')
plt.legend()
plt.tight_layout()
plt.show()
print('Segmentation onset rule result:')
print(json.dumps(onset, indent=2))

In [ ]:
# 7. Download the complete result bundle (no Google Drive)
RESULT_ZIP = pathlib.Path('/content/segmentation_probe_results.zip')
assert RESULT_ZIP.is_file(), RESULT_ZIP
print(f'Downloading {RESULT_ZIP.name} ({RESULT_ZIP.stat().st_size / 1_000_000:.1f} MB)')
files.download(str(RESULT_ZIP))